# Hands-On AI: Quickstart

A short tour of the toolkit. Each module is the same shape underneath: one call to a model, wrapped in plain Python.

> **Running on Colab?** A hosted notebook cannot reach an Ollama on your own laptop, so you need a provider it can reach over the internet (for example, an Ollama server with a bearer key that your educator hosts). The setup cell below covers it.
>
> **Running locally** (Jupyter / JupyterLab / VS Code) with Ollama installed? You can skip the setup cell entirely.

## 1. Install

In [ ]:
!pip install -q hands-on-ai

## 2. Connect to a model

**Local with Ollama running?** Skip this cell. Hands-On AI uses `http://localhost:11434` automatically.

**On Colab or a hosted notebook?** Fill in a provider you can reach. Any OpenAI-compatible provider works (a hosted Ollama, OpenAI, Groq, OpenRouter, and so on).

In [ ]:
import os, getpass

os.environ["HANDS_ON_AI_SERVER"] = "https://ollama.your-school.edu"  # your provider URL
os.environ["HANDS_ON_AI_MODEL"]  = "llama3"                          # a model the provider has
os.environ["HANDS_ON_AI_API_KEY"] = getpass.getpass("API key (leave blank if none): ")

## 3. Your first response

The one call everything else is built on:

In [ ]:
from hands_on_ai.chat import get_response

print(get_response("Explain photosynthesis like I am 10."))

## 4. Personality bots

A "bot" is just `get_response` with a fixed system prompt. Same call, different character:

In [ ]:
from hands_on_ai.chat import pirate_bot, teacher_bot

print(pirate_bot("Tell me about the ocean."))
print()
print(teacher_bot("What is a variable in programming?"))

## 5. Memory with `Conversation`

An LLM has no memory of its own: every `get_response` call is independent. To make a bot that remembers, you resend the conversation each turn. `Conversation` does that bookkeeping for you.

In [ ]:
from hands_on_ai.chat import Conversation

chat = Conversation(system="You are a friendly tutor.")
chat.ask("My name is Sam.")
print(chat.ask("What is my name?"))      # it remembers "Sam"
print("tokens used so far:", chat.total_tokens)

## 6. A tool-using agent

An agent is a loop: the model decides which tool to call, your Python runs it, and the result goes back to the model. Here we register the built-in calculator and let the agent use it.

(Small local models vary in how reliably they call tools. If the answer looks off, try a more capable model.)

In [ ]:
from hands_on_ai.agent import run_agent, register_calculator_tool

register_calculator_tool()
print(run_agent("What is 23 * 19? Use the calculator tool."))

## 7. A multi-step workflow

A workflow is a folder of numbered stages. The model walks them in order, writing a readable file at each step, and you review between steps. Here we build a tiny two-stage pipeline in code.

In [ ]:
from pathlib import Path
from hands_on_ai.workflow import Pipeline, init_workspace

init_workspace("demo_workflow", ["facts", "explain"])
Path("demo_workflow/stages/01_facts/CONTEXT.md").write_text(
    "List 3 short, accurate facts about the Moon. Numbered list only.")
Path("demo_workflow/stages/02_explain/CONTEXT.md").write_text(
    "Using the facts provided, write one friendly sentence for a child.")

pipe = Pipeline("demo_workflow")

pipe.run_next()   # runs stage 1 and stops so you can review it
print("--- Stage 1 (facts) ---")
print(Path("demo_workflow/stages/01_facts/output/output.md").read_text())

In [ ]:
pipe.run_next()   # runs stage 2 using the facts from stage 1
print("--- Stage 2 (explanation) ---")
print(Path("demo_workflow/stages/02_explain/output/output.md").read_text())

## Where to go next

- **Concepts:** [Understanding Chat & Bots](https://michael-borck.github.io/hands-on-ai/chat-concepts/), [RAG](https://michael-borck.github.io/hands-on-ai/rag-llm-explain/), [Agents](https://michael-borck.github.io/hands-on-ai/agent-concepts/), [Workflows](https://michael-borck.github.io/hands-on-ai/workflow-guide/)
- **Build something:** the [project gallery](https://michael-borck.github.io/hands-on-ai/projects/)
- **Why it is small on purpose:** [Our Approach](https://michael-borck.github.io/hands-on-ai/philosophy/)

The model is the only "AI" here. Everything else is code you can open and read.